**Project 12: Gym & Fitness Center Membership Analytics**



In [ ]:
# ==========================================================
# PROJECT 12: GYM & FITNESS CENTER MEMBERSHIP ANALYTICS
# HCL DATA ANALYSIS TRAINING
#
# NAME-UTKARSH GOLA
# ADMISSION No-2026B02310106
# GitHub Link-https://github.com/utkarshgola1/Python_Project
#
# Libraries: Pandas and NumPy
# ==========================================================


# STEP 1: IMPORT LIBRARIES

import pandas as pd
import numpy as np


# STEP 2: DOWNLOAD DATASET FROM GITHUB

print("Loading Gym Membership Dataset...")

github_link = "https://raw.githubusercontent.com/utkarshgola1/Python_Project/main/gym_membership.csv"

df = pd.read_csv(github_link)

print("Dataset loaded successfully!")

print("\nOriginal Dataset:")
print(df.head())

print("\nTotal Rows and Columns:")
print(df.shape)


Loading Gym Membership Dataset...
Dataset loaded successfully!

Original Dataset:
  Checkin_ID Member_ID Membership_Tier  Gender         Checkin_Time  \
0     G00000     M0316        Platinum   Other  2026-05-01 22:00:00   
1     G00001     M0234           Basic    Male  2025-05-01 07:00:00   
2     G00002     M0094            Gold  Female  2026-10-14 17:00:00   
3     G00003     M0204            Gold   Other  2025-10-09 15:00:00   
4     G00004     M0281           Basic    Male  2025-04-11 09:00:00   

   Workout_Duration_Mins Fitness_Class  Monthly_Fee  
0                   30.0          HIIT          999  
1                   71.0          Spin         2499  
2                  129.0          Spin         1499  
3                    NaN          Spin         1499  
4                  107.0          Spin          999  

Total Rows and Columns:
(862, 8)


In [ ]:
# STEP 3: CHECK DATA QUALITY

print("\nMISSING VALUES:")
print(df.isnull().sum())

print("\nDUPLICATE CHECK-IN RECORDS:")
print(df["Checkin_ID"].duplicated().sum())

print("\nDATA TYPES:")
print(df.dtypes)



MISSING VALUES:
Checkin_ID                 0
Member_ID                  0
Membership_Tier            0
Gender                     0
Checkin_Time               0
Workout_Duration_Mins     21
Fitness_Class            155
Monthly_Fee                0
dtype: int64

DUPLICATE CHECK-IN RECORDS:
12

DATA TYPES:
Checkin_ID                object
Member_ID                 object
Membership_Tier           object
Gender                    object
Checkin_Time              object
Workout_Duration_Mins    float64
Fitness_Class             object
Monthly_Fee                int64
dtype: object


In [ ]:
# STEP 4: REMOVE DUPLICATE CHECK-IN RECORDS

df = df.drop_duplicates(
    subset="Checkin_ID",
    keep="first"
)

print("\nDuplicate records removed.")



Duplicate records removed.


In [ ]:
# STEP 5: CLEAN MISSING WORKOUT DURATIONS

# We will use the median duration of each membership tier.

tiers = df["Membership_Tier"].unique()

for tier in tiers:

    median_value = df[
        df["Membership_Tier"] == tier
    ]["Workout_Duration_Mins"].median()

    df.loc[
        (df["Membership_Tier"] == tier) &
        (df["Workout_Duration_Mins"].isnull()),
        "Workout_Duration_Mins"
    ] = median_value


# Fill remaining missing durations with overall median.

overall_median = df["Workout_Duration_Mins"].median()

df["Workout_Duration_Mins"] = df[
    "Workout_Duration_Mins"
].fillna(overall_median)


# Missing fitness classes will be labelled Unknown.

df["Fitness_Class"] = df["Fitness_Class"].fillna("Unknown")

print("\nMissing values cleaned.")


Missing values cleaned.


In [ ]:
# STEP 6: CONVERT CHECK-IN TIME

df["Checkin_Time"] = pd.to_datetime(
    df["Checkin_Time"],
    errors="coerce"
)

# Remove records containing invalid date/time values.

df = df.dropna(subset=["Checkin_Time"])

# Extract weekday, hour and month.

df["Weekday"] = df["Checkin_Time"].dt.day_name()

df["Hour"] = df["Checkin_Time"].dt.hour

df["Month"] = df["Checkin_Time"].dt.strftime("%Y-%m")

print("\nDate and time conversion completed.")


Date and time conversion completed.


In [ ]:
# STEP 7: TOTAL CHECK-INS AND AVERAGE WORKOUT DURATION

total_checkins = len(df)

average_duration = df["Workout_Duration_Mins"].mean()

print("\nTOTAL CHECK-INS:", total_checkins)

print(
    "AVERAGE WORKOUT DURATION:",
    round(average_duration, 2),
    "minutes"
)



TOTAL CHECK-INS: 850
AVERAGE WORKOUT DURATION: 72.41 minutes


In [ ]:
# STEP 8: MEMBERSHIP TIER COMPARISON

total_visits = df.groupby(
    "Membership_Tier"
)["Checkin_ID"].count()

unique_members = df.groupby(
    "Membership_Tier"
)["Member_ID"].nunique()

average_workout = df.groupby(
    "Membership_Tier"
)["Workout_Duration_Mins"].mean()

tier_analysis = pd.DataFrame({
    "Total_Checkins": total_visits,
    "Unique_Members": unique_members,
    "Average_Workout": average_workout
})

tier_analysis["Average_Visits_Per_Member"] = (
    tier_analysis["Total_Checkins"] /
    tier_analysis["Unique_Members"]
)

print("\nMEMBERSHIP TIER ANALYSIS:")

print(tier_analysis.round(2))



MEMBERSHIP TIER ANALYSIS:
                 Total_Checkins  Unique_Members  Average_Workout  \
Membership_Tier                                                    
Basic                       415             242            71.82   
Gold                        293             212            74.52   
Platinum                    142             122            69.78   

                 Average_Visits_Per_Member  
Membership_Tier                             
Basic                                 1.71  
Gold                                  1.38  
Platinum                              1.16  


In [ ]:
# STEP 9: FIND PEAK GYM HOURS

hour_visits = df["Hour"].value_counts().sort_index()

print("\nCHECK-INS BY HOUR:")

print(hour_visits)

peak_hour = hour_visits[
    hour_visits == hour_visits.max()
]

print("\nPEAK GYM HOUR:")

print(peak_hour)


# FIND BUSIEST WEEKDAYS

days = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

day_visits = df["Weekday"].value_counts()

day_visits = day_visits.reindex(
    days,
    fill_value=0
)

print("\nCHECK-INS BY WEEKDAY:")

print(day_visits)

busiest_day = day_visits[
    day_visits == day_visits.max()
]

print("\nBUSIEST DAY:")

print(busiest_day)



CHECK-INS BY HOUR:
Hour
5     46
6     46
7     63
8     44
9     37
10    46
11    43
12    44
13    63
14    45
15    38
16    40
17    40
18    51
19    36
20    36
21    51
22    39
23    42
Name: count, dtype: int64

PEAK GYM HOUR:
Hour
7     63
13    63
Name: count, dtype: int64

CHECK-INS BY WEEKDAY:
Weekday
Monday       127
Tuesday      114
Wednesday    126
Thursday     125
Friday       101
Saturday     137
Sunday       120
Name: count, dtype: int64

BUSIEST DAY:
Weekday
Saturday    137
Name: count, dtype: int64


In [ ]:
# STEP 10: FITNESS CLASS ATTENDANCE

# Unknown classes are excluded from named class rankings.

known_classes = df[
    df["Fitness_Class"] != "Unknown"
]

class_attendance = known_classes[
    "Fitness_Class"
].value_counts()

print("\nFITNESS CLASS ATTENDANCE:")

print(class_attendance)

print("\nMOST POPULAR FITNESS CLASS:")

print(class_attendance.head(1))


FITNESS CLASS ATTENDANCE:
Fitness_Class
Strength    151
Spin        143
Zumba       136
HIIT        135
Yoga        132
Name: count, dtype: int64

MOST POPULAR FITNESS CLASS:
Fitness_Class
Strength    151
Name: count, dtype: int64


In [ ]:
# STEP 11: COMPARE MALE AND FEMALE WORKOUT DURATION

gender_data = df[
    (df["Gender"] == "Male") |
    (df["Gender"] == "Female")
]

gender_analysis = gender_data.groupby(
    "Gender"
)["Workout_Duration_Mins"].mean()

print("\nAVERAGE WORKOUT DURATION BY GENDER:")

print(gender_analysis.round(2))



AVERAGE WORKOUT DURATION BY GENDER:
Gender
Female    73.02
Male      73.05
Name: Workout_Duration_Mins, dtype: float64


In [ ]:
# STEP 12: MONTHLY MEMBERSHIP REVENUE BY TIER

# Assumption:
# Each member contributes one monthly membership fee
# for each recorded month and membership tier.
#
# If multiple fees are recorded for the same member,
# their median fee is used as the representative fee.

member_fee = df.groupby(
    ["Month", "Membership_Tier", "Member_ID"]
)["Monthly_Fee"].median()

member_fee = member_fee.reset_index()

monthly_revenue = member_fee.groupby(
    ["Month", "Membership_Tier"]
)["Monthly_Fee"].sum()

monthly_revenue = monthly_revenue.unstack(
    fill_value=0
)

print("\nMONTHLY MEMBERSHIP REVENUE:")

print(monthly_revenue)


MONTHLY MEMBERSHIP REVENUE:
Membership_Tier    Basic     Gold  Platinum
Month                                      
2025-01          22489.0  21987.0   10994.0
2025-02          25983.0  23983.0     999.0
2025-03          38478.0  20235.0   17491.0
2025-04          30981.0  22485.0   12992.0
2025-05          14990.0  24986.0    6996.0
2025-06          31980.0  13491.0    9993.0
2025-07          31482.0  21488.0    7495.0
2025-08          38479.0  12492.0    9495.0
2025-09          27732.0  19490.0    8494.0
2025-10          17987.0  16489.0    6996.0
2025-11          30480.0  14989.0    9995.0
2025-12          35480.0  37479.0    8995.0
2026-01          30731.0  26986.0   13492.0
2026-02          35728.0  10994.0   12993.0
2026-03          41478.0  14488.0   12493.0
2026-04          29981.0  22987.0   13992.0
2026-05          36978.0  10491.0   19988.0
2026-06          36729.0  24485.0    9494.0
2026-07          39475.0  27986.0   22488.0
2026-08          30981.0  34979.0    6496.0
202

In [ ]:
# STEP 13: FIND LOW-ENGAGEMENT MEMBERS

# Members having fewer than 2 visits in a month.

monthly_visits = df.groupby(
    ["Member_ID", "Month"]
).size()

monthly_visits = monthly_visits.reset_index(
    name="Total_Visits"
)

low_engagement = monthly_visits[
    monthly_visits["Total_Visits"] < 2
]

print("\nLOW-ENGAGEMENT MEMBER RECORDS:")

print(low_engagement)

print(
    "\nTotal low-engagement member-months:",
    len(low_engagement)
)

print(
    "Distinct low-engagement members:",
    low_engagement["Member_ID"].nunique()
)


LOW-ENGAGEMENT MEMBER RECORDS:
    Member_ID    Month  Total_Visits
0       M0001  2026-09             1
1       M0002  2025-03             1
2       M0003  2025-04             1
3       M0003  2025-12             1
4       M0003  2026-07             1
..        ...      ...           ...
798     M0398  2025-03             1
800     M0398  2026-01             1
802     M0399  2025-04             1
803     M0399  2025-09             1
804     M0399  2026-10             1

[761 rows x 3 columns]

Total low-engagement member-months: 761
Distinct low-engagement members: 345


In [ ]:
# STEP 14: NUMPY STATISTICAL ANALYSIS

durations = df[
    "Workout_Duration_Mins"
].dropna().to_numpy()

median = np.median(durations)

variance = np.var(durations)

quartiles = np.percentile(
    durations,
    [25, 50, 75]
)

Q1 = quartiles[0]

Q2 = quartiles[1]

Q3 = quartiles[2]

print("\nWORKOUT STATISTICS:")

print("Median:", round(median, 2))

print("Variance:", round(variance, 2))

print("Q1:", round(Q1, 2))

print("Q2:", round(Q2, 2))

print("Q3:", round(Q3, 2))


# FIND UNUSUAL WORKOUT SESSIONS USING IQR

IQR = Q3 - Q1

lower_limit = Q1 - (1.5 * IQR)

upper_limit = Q3 + (1.5 * IQR)

print("\nLOWER LIMIT:", round(lower_limit, 2))

print("UPPER LIMIT:", round(upper_limit, 2))

outliers = df[
    (df["Workout_Duration_Mins"] < lower_limit) |
    (df["Workout_Duration_Mins"] > upper_limit)
]

print("\nUNUSUAL WORKOUT SESSIONS:")

print(
    outliers[
        ["Checkin_ID", "Member_ID", "Workout_Duration_Mins"]
    ]
)



WORKOUT STATISTICS:
Median: 71.0
Variance: 974.64
Q1: 47.0
Q2: 71.0
Q3: 98.75

LOWER LIMIT: -30.62
UPPER LIMIT: 176.38

UNUSUAL WORKOUT SESSIONS:
Empty DataFrame
Columns: [Checkin_ID, Member_ID, Workout_Duration_Mins]
Index: []


In [ ]:
# STEP 15: CREATE WORKOUT INTENSITY LABELS

# Short: 45 minutes or less
# Moderate: 46 to 90 minutes
# Extended: More than 90 minutes

df["Workout_Intensity"] = np.where(
    df["Workout_Duration_Mins"] <= 45,
    "Short",
    np.where(
        df["Workout_Duration_Mins"] <= 90,
        "Moderate",
        "Extended"
    )
)

print("\nWORKOUT INTENSITY:")

print(df["Workout_Intensity"].value_counts())


# STEP 16: FACILITY MANAGEMENT RECOMMENDATIONS

print("\nFACILITY MANAGEMENT RECOMMENDATIONS")

print("1. Arrange additional staff during peak gym hours.")

print("2. Provide extra support on the busiest weekday.")

print("3. Review the schedule and capacity of popular fitness classes.")

print("4. Contact members who visit fewer than twice a month.")

print("5. Review unusually short and long workout sessions.")



WORKOUT INTENSITY:
Workout_Intensity
Moderate    380
Extended    268
Short       202
Name: count, dtype: int64

FACILITY MANAGEMENT RECOMMENDATIONS
1. Arrange additional staff during peak gym hours.
2. Provide extra support on the busiest weekday.
3. Review the schedule and capacity of popular fitness classes.
4. Contact members who visit fewer than twice a month.
5. Review unusually short and long workout sessions.


In [ ]:
# STEP 17: SAVE CLEANED DATASET

output_file = "Gym_Membership_Cleaned.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nCLEANING COMPLETED SUCCESSFULLY!")

print("\nFinal Dataset Shape:", df.shape)

print("\nRemaining Missing Values:")

print(df.isnull().sum())

print("\nCleaned file saved as:", output_file)


# STEP 18: AUTOMATICALLY DOWNLOAD CSV

try:

    from google.colab import files

    files.download(output_file)

    print("\nCSV DOWNLOADED SUCCESSFULLY!")

except ImportError:

    print("\nCSV saved in your current Python folder.")

    print("Open that folder to access your cleaned dataset.")


print("\nPROJECT COMPLETED!")



CLEANING COMPLETED SUCCESSFULLY!

Final Dataset Shape: (850, 12)

Remaining Missing Values:
Checkin_ID               0
Member_ID                0
Membership_Tier          0
Gender                   0
Checkin_Time             0
Workout_Duration_Mins    0
Fitness_Class            0
Monthly_Fee              0
Weekday                  0
Hour                     0
Month                    0
Workout_Intensity        0
dtype: int64

Cleaned file saved as: Gym_Membership_Cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


CSV DOWNLOADED SUCCESSFULLY!

PROJECT COMPLETED!
